# Part 1: Bridge Data Preparation and PGA Assignment

## What this file does
This part of the workflow prepares the bridge inventory dataset for later vulnerability and damage analysis. The goal is to start with the California National Bridge Inventory (NBI), clean the coordinate information, convert bridge locations into usable geospatial points, and assign peak ground acceleration (PGA) values from the earthquake raster.

## Why this step is important
Bridge vulnerability analysis depends on combining bridge attributes with hazard intensity at each bridge location. Before any HAZUS fragility, SVI scoring, or machine learning can be performed, the bridge inventory must be cleaned and linked to the spatial hazard data correctly.

## Main tasks completed
- Loaded the California bridge inventory file
- Cleaned latitude and longitude fields from the NBI dataset
- Converted NBI coordinate values into decimal degrees
- Created a GeoDataFrame of bridge locations
- Read the PGA raster from the ShakeMap output
- Sampled raster PGA values at bridge locations
- Attached the sampled PGA values to each bridge
- Exported the prepared dataset for the next stages of the project

## Output
The final output from this step is a cleaned bridge dataset with:
- bridge identifiers
- usable latitude and longitude
- geometry for mapping
- assigned PGA values

This file forms the base input for later:
- HAZUS bridge classification
- fragility-based damage probability calculation
- SVI scoring
- machine learning modeling

## How to use this notebook

Use this notebook as the main starting point for the core reproducible workflow. It takes the raw California bridge inventory and the USGS ShakeMap PGA raster, cleans bridge coordinates, samples hazard intensity, and creates the bridge-level hazard table used by the rest of the project.

**Inputs**
- `data/CA25.txt`
- `data/pga_mean.flt`
- `data/pga_mean.hdr`

**Output**
- `data/processed/pga_nbi_bridge.csv`

**Why this step matters**
This is the bridge between the raw inventory and the hazard model. If this table is wrong, every downstream fragility, vulnerability, and ML result will also be wrong.

**Next notebook**
After this notebook, run `HAZUS.ipynb`.


In [ ]:
from runtime_checks import ensure_packages, ensure_supported_runtime

ensure_supported_runtime()
ensure_packages([
    "numpy",
    "pandas",
    "matplotlib",
    "scipy",
    "sklearn",
    "rasterio",
    "geopandas",
    "shapely",
    "seaborn",
    "rasterstats",
])

print("Environment check passed. Use a local Python kernel from the project environment.")


In [ ]:
from pathlib import Path

from project_paths import build_paths

PATHS = build_paths()
print("Project root:", PATHS["PROJECT_ROOT"])
print("Data dir:", PATHS["DATA_DIR"])
print("Processed dir:", PATHS["PROCESSED_DIR"])


In [ ]:
print("Install project dependencies first: pip install -r requirements.txt")


In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
import rasterio

from project_paths import build_paths, require_paths

PATHS = build_paths()
NBI_FILE = PATHS["NBI_FILE"]
RASTER_PATH = PATHS["PGA_RASTER"]
OUT_ALL_CSV = PATHS["PGA_BRIDGE_CSV"]

require_paths(PATHS, ["NBI_FILE", "PGA_RASTER"])

def nbi_ddmmss_to_decimal(x, is_lon=False):
    if pd.isna(x):
        return np.nan
    s = str(x).strip().replace(",", "")
    if not s.isdigit():
        return np.nan
    try:
        if is_lon:
            if len(s) < 9:
                s = s.zfill(9)
            deg = int(s[:3])
            mins = int(s[3:5])
            secs = float(s[5:]) / 100.0
            val = deg + mins / 60 + secs / 3600
            return -val
        if len(s) < 8:
            s = s.zfill(8)
        deg = int(s[:2])
        mins = int(s[2:4])
        secs = float(s[4:]) / 100.0
        val = deg + mins / 60 + secs / 3600
        return val
    except Exception:
        return np.nan

df = pd.read_csv(NBI_FILE, low_memory=False)

LAT_COL = "LAT_016"
LON_COL = "LONG_017"
ID_COL = "STRUCTURE_NUMBER_008"

required_cols = [ID_COL, LAT_COL, LON_COL]
missing_cols = [c for c in required_cols if c not in df.columns]
if missing_cols:
    raise ValueError(f"Missing required columns: {missing_cols}")

df["lat_raw"] = df[LAT_COL].astype(str).str.replace(",", "", regex=False).str.strip()
df["lon_raw"] = df[LON_COL].astype(str).str.replace(",", "", regex=False).str.strip()
df["latitude"] = df["lat_raw"].apply(lambda x: nbi_ddmmss_to_decimal(x, is_lon=False))
df["longitude"] = df["lon_raw"].apply(lambda x: nbi_ddmmss_to_decimal(x, is_lon=True))
df = df.dropna(subset=["latitude", "longitude"]).copy()

with rasterio.open(RASTER_PATH) as src:
    nodata = src.nodata
    coords = list(zip(df["longitude"], df["latitude"]))
    sampled = list(src.sample(coords))

raw_vals = [val[0] if val is not None else np.nan for val in sampled]
df["pga_raw"] = pd.to_numeric(raw_vals, errors="coerce")
if nodata is not None:
    df.loc[df["pga_raw"] == nodata, "pga_raw"] = np.nan

df.loc[df["pga_raw"] >= 900, "pga_raw"] = np.nan
df["pga"] = np.exp(df["pga_raw"])
df.loc[~np.isfinite(df["pga"]), "pga"] = np.nan
df["join_id"] = df[ID_COL].astype(str).str.strip().str.upper()

df.to_csv(OUT_ALL_CSV, index=False)

print("Rows with usable coordinates:", len(df))
print("Latitude range:", df["latitude"].min(), "to", df["latitude"].max())
print("Longitude range:", df["longitude"].min(), "to", df["longitude"].max())
print(df[[ID_COL, "latitude", "longitude", "pga_raw", "pga"]].head())
print(df["pga"].describe())
print("Saved:", OUT_ALL_CSV)


# Part 1 Plots: Coordinate and PGA Verification

These plots are used to verify that the bridge inventory was cleaned correctly and that PGA values were sampled reasonably from the raster.

The first plot shows the spatial distribution of bridge coordinates after conversion to decimal degrees. This helps confirm that the bridge locations fall within California.

The second plot shows the distribution of converted PGA values. This helps check whether the sampled hazard values are positive and within a realistic range after transforming the raster from log scale.

In [ ]:
import matplotlib.pyplot as plt

plot_df = df.dropna(subset=["latitude", "longitude"]).copy()
pga_plot_df = df.dropna(subset=["pga"]).copy()

plt.figure(figsize=(8, 6))
plt.scatter(plot_df["longitude"], plot_df["latitude"], s=5)
plt.xlabel("Longitude")
plt.ylabel("Latitude")
plt.title("Bridge Locations in California")
plt.grid(True)
plt.show()

plt.figure(figsize=(8, 6))
plt.hist(pga_plot_df["pga"], bins=40)
plt.xlabel("PGA")
plt.ylabel("Count")
plt.title("Distribution of Sampled PGA Values")
plt.grid(True)
plt.show()

## Additional check

This scatter plot shows bridge location colored by PGA, which helps visually confirm that the assigned hazard values vary spatially across the study region.

In [ ]:
plt.figure(figsize=(8, 6))
sc = plt.scatter(
    pga_plot_df["longitude"],
    pga_plot_df["latitude"],
    c=pga_plot_df["pga"],
    s=8
)
plt.xlabel("Longitude")
plt.ylabel("Latitude")
plt.title("Bridge Locations Colored by PGA")
plt.grid(True)
plt.colorbar(sc, label="PGA")
plt.show()

## Inference

The results from Part 1 confirm that the bridge inventory was cleaned and georeferenced successfully. The latitude and longitude ranges fall within California, and the bridge location scatter plot shows a spatial pattern consistent with the state’s transportation network. This indicates that the coordinate conversion from the original NBI format to decimal degrees worked correctly.

The sampled PGA values also appear reasonable after converting the raster from log scale back to standard PGA values. The histogram shows a strongly right-skewed distribution, with most bridges experiencing relatively low PGA and a smaller number of bridges exposed to higher shaking levels. This is expected in earthquake hazard data, where strong shaking is usually concentrated around the event source and decays with distance.

The spatial plot colored by PGA further supports this interpretation. Higher PGA values are concentrated in Southern California, especially near the likely Northridge earthquake influence zone, while lower values dominate elsewhere. Overall, this step successfully links bridge inventory data with hazard intensity and creates a reliable base dataset for the following stages of HAZUS classification, fragility analysis, SVI computation, and machine learning.